# Self-Supervised Learning: SimCLR, MoCo, MAE

**Module 4 - Notebook 6** | Skillsoft Track 2: DL Programmer

## Objectives
- SimCLR implementation on CIFAR-10
- Linear probe evaluation
- MoCo overview
- MAE (Masked Autoencoder) concept

In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. SimCLR: Contrastive Learning Framework

In [ ]:
# SimCLR Augmentations
class SimCLRAugment:
    def __init__(self, size=32):
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([
                transforms.ColorJitter(0.8, 0.8, 0.8, 0.2)
            ], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.GaussianBlur(kernel_size=3),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
        ])
    
    def __call__(self, x):
        return self.transform(x), self.transform(x)

# NT-Xent Loss
def nt_xent_loss(z_i, z_j, temperature=0.5):
    """Normalized Temperature-scaled Cross Entropy Loss."""
    batch_size = z_i.size(0)
    
    # Normalize
    z_i = F.normalize(z_i, dim=1)
    z_j = F.normalize(z_j, dim=1)
    
    # Similarity matrix
    representations = torch.cat([z_i, z_j], dim=0)  # 2N x D
    similarity = F.cosine_similarity(representations.unsqueeze(1), representations.unsqueeze(0), dim=2)
    
    # Mask out self-similarity
    mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z_i.device)
    similarity = similarity.masked_fill(mask, -1e9)
    
    # Positive pairs: (i, i+N) and (i+N, i)
    positives = torch.cat([
        torch.diag(similarity, batch_size),
        torch.diag(similarity, -batch_size)
    ], dim=0)
    
    # Loss
    logits = similarity / temperature
    labels = torch.arange(2 * batch_size, device=z_i.device)
    labels[:batch_size] += batch_size
    labels[batch_size:] -= batch_size
    
    loss = F.cross_entropy(logits, labels)
    return loss

# SimCLR Model
class SimCLR(nn.Module):
    def __init__(self, base_encoder, projection_dim=128):
        super().__init__()
        self.encoder = base_encoder
        # Projection head
        self.projection = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, projection_dim)
        )
    
    def forward(self, x):
        h = self.encoder(x)
        z = self.projection(h)
        return h, z

# ResNet-18 encoder (CIFAR version)
from torchvision.models import resnet18
base_encoder = resnet18(num_classes=512)
base_encoder.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
base_encoder.maxpool = nn.Identity()

model = SimCLR(base_encoder).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)

# Data
train_dataset = torchvision.datasets.CIFAR10(
    './data', train=True, download=True,
    transform=SimCLRAugment()
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, drop_last=True)

# Train a few epochs
model.train()
for epoch in range(3):
    total_loss = 0
    for (x_i, x_j), _ in train_loader:
        x_i, x_j = x_i.to(device), x_j.to(device)
        optimizer.zero_grad()
        _, z_i = model(x_i)
        _, z_j = model(x_j)
        loss = nt_xent_loss(z_i, z_j)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}: NT-Xent Loss = {total_loss/len(train_loader):.4f}")

## 2. Linear Probe Evaluation

In [ ]:
# Freeze encoder, train linear classifier
for param in model.encoder.parameters():
    param.requires_grad = False

class LinearProbe(nn.Module):
    def __init__(self, encoder, num_classes=10):
        super().__init__()
        self.encoder = encoder
        self.fc = nn.Linear(512, num_classes)
    
    def forward(self, x):
        with torch.no_grad():
            h = self.encoder(x)
        return self.fc(h)

probe = LinearProbe(model.encoder).to(device)
optimizer_probe = torch.optim.Adam(probe.fc.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

# Standard CIFAR-10 train transform
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

probe_train_dataset = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=train_transform)
probe_train_loader = DataLoader(probe_train_dataset, batch_size=256, shuffle=True, num_workers=2)

probe_test_dataset = torchvision.datasets.CIFAR10('./data', train=False, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ]))
probe_test_loader = DataLoader(probe_test_dataset, batch_size=256, shuffle=False, num_workers=2)

def train_probe(epoch):
    probe.train()
    for data, target in probe_train_loader:
        data, target = data.to(device), target.to(device)
        optimizer_probe.zero_grad()
        output = probe(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer_probe.step()

def eval_probe():
    probe.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for data, target in probe_test_loader:
            data, target = data.to(device), target.to(device)
            output = probe(data)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)
    return correct / total

for epoch in range(10):
    train_probe(epoch)
    acc = eval_probe()
    print(f"Epoch {epoch}: Linear Probe Acc = {acc:.4f}")

## 3. MoCo (Momentum Contrast) Overview

In [ ]:
moco_overview = """
MoCo (Momentum Contrast) - Key Ideas:

1. MOMENTUM ENCODER
   - Key encoder updated as moving average of query encoder
   - theta_k = m * theta_k + (1-m) * theta_q
   - m=0.999 typically
   - Provides consistent dictionary for contrastive learning

2. QUEUE
   - Large dictionary of negative samples (65K+)
   - Enqueue current batch keys, dequeue oldest
   - Decouples dictionary size from batch size

3. SHUFFLE BN
   - Shuffle batch across GPUs before BN
   - Prevents information leakage in BN stats

MoCo v2 Improvements:
- MLP projection head (2-layer)
- Stronger augmentations (SimCLR-style)
- Cosine annealing LR
- Longer training (200-800 epochs)

MoCo v3 (Vision Transformers):
- ViT backbone
- No queue needed (uses large batch)
- Better for transformer architectures
"""

print(moco_overview)

## 4. MAE (Masked Autoencoder) Concept

In [ ]:
mae_concept = """
MAE (Masked Autoencoder) - Key Ideas:

1. HIGH MASKING RATIO (75%)
   - Only 25% patches visible to encoder
   - Forces learning of semantic representations
   - Much more efficient than 15% in BERT

2. ASYMMETRIC ARCHITECTURE
   - Encoder: processes only visible patches (small, fast)
   - Decoder: lightweight, reconstructs all patches
   - Decoder only used during pre-training

3. PATCH-BASED
   - Image divided into 16x16 patches
   - Linear projection + positional embedding
   - Standard ViT backbone

4. RECONSTRUCTION TARGET
   - Predict pixel values (normalized per patch)
   - MSE loss on masked patches only
   - Simpler than contrastive learning

IMPLEMENTATION OUTLINE:

class MAE(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3,
                 embed_dim=768, depth=12, num_heads=12,
                 decoder_embed_dim=512, decoder_depth=8, decoder_num_heads=16):
        # Patch embedding
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches+1, embed_dim))
        
        # Encoder (only visible patches)
        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads, mlp_ratio=4, qkv_bias=True)
            for _ in range(depth)
        ])
        
        # Decoder
        self.decoder_embed = nn.Linear(embed_dim, decoder_embed_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, num_patches+1, decoder_embed_dim))
        self.decoder_blocks = nn.ModuleList([
            Block(decoder_embed_dim, decoder_num_heads, mlp_ratio=4, qkv_bias=True)
            for _ in range(decoder_depth)
        ])
        self.decoder_pred = nn.Linear(decoder_embed_dim, patch_size**2 * in_chans)
    
    def forward(self, x, mask_ratio=0.75):
        # Patchify
        x = self.patch_embed(x)
        x = x + self.pos_embed[:, 1:, :]
        
        # Random masking
        x, mask, ids_restore = self.random_masking(x, mask_ratio)
        
        # Encoder
        x = self.encoder(x)
        
        # Decoder
        x = self.decoder(x, ids_restore)
        pred = self.decoder_pred(x)
        
        # MSE loss on masked patches
        loss = self.forward_loss(x, pred, mask)
        return loss, pred, mask
"""

print(mae_concept)

In [ ]:
# Executable MAE: Masked patch reconstruction (simplified)
class SimpleMAE(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()
        # Simpler version: linear encoder for patches
        self.encoder = nn.Linear(28*28, 128)
        self.decoder = nn.Linear(128, 28*28)

    def forward(self, x):
        x_flat = x.view(x.size(0), -1)
        encoded = F.relu(self.encoder(x_flat))
        return self.decoder(encoded)

mae_exec = SimpleMAE().to(device)
mae_opt = torch.optim.Adam(mae_exec.parameters(), lr=1e-3)

# Quick MAE demo on MNIST (reconstruction task)
mae_exec.train()
for epoch in range(2):  # Demo: 2 epochs
    total_loss = 0
    for data, _ in train_loader:
        data = data.to(device)
        mae_opt.zero_grad()
        recon = mae_exec(data)
        loss = F.mse_loss(recon, data.view(data.size(0), -1))
        loss.backward()
        mae_opt.step()
        total_loss += loss.item()
    print(f"MAE Epoch {epoch}: Reconstruction Loss = {total_loss/len(train_loader):.4f}")

print("MAE executable complete")
print(f"Trainable params: {sum(p.numel() for p in mae_exec.parameters() if p.requires_grad)}")